In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness, Isomap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import pairwise_distances

from sklearn import datasets

In [ ]:
n_samples = 100
n_features = 20
seed = 42

gaus_homosk_blobs = datasets.make_blobs(n_samples=n_samples,n_features=n_features, random_state=seed)

sample_X = gaus_homosk_blobs[0]

In [ ]:
from scipy.stats import rankdata

In [ ]:
import numpy as np


In [ ]:
dist_X = pairwise_distances(sample_X)
rankdata(dist_X, axis=1, method="ordinal").astype(int)-1

In [ ]:
# X_pca = PCA(n_components=2).fit_transform(sample_X)
X_pca = TSNE(n_components=2, perplexity=1, random_state=0,method='barnes_hut').fit_transform(sample_X)
dist_X_pca = pairwise_distances(X_pca)

In [ ]:
R1 = np.array([np.argsort(np.argsort(row)) for row in dist_X])
R2 = np.array([np.argsort(np.argsort(row)) for row in dist_X_pca])

In [ ]:
Q = np.bincount((R1 * R1.shape[0] + R2).ravel(), minlength=R1.shape[0] * R1.shape[0]).reshape(R1.shape[0],R1.shape[0])

In [ ]:
R1, R2

In [ ]:
Q

In [ ]:
coranking_matrix(R1, R2)

In [ ]:
np.fill_diagonal(dist_X, np.inf)
np.fill_diagonal(dist_X_pca, np.inf)

R1 = np.array([np.argsort(np.argsort(row)) for row in dist_X])
R2 = np.array([np.argsort(np.argsort(row)) for row in dist_X_pca])
Qi = np.bincount((R1 * R1.shape[0] + R2).ravel(), minlength=R1.shape[0] * R1.shape[0]).reshape(R1.shape[0],R1.shape[0])

In [ ]:
Q[0:,:0]

In [ ]:
Q[1:,1:], Qi[:-1,:-1]

In [ ]:
_trustworthiness(Qi)[:5]

In [ ]:
compute_trustworthiness(Qi)[:5]

In [ ]:
[trustworthiness(sample_X,X_pca,n_neighbors=v) for v in range(1,6)]

In [ ]:
ka = np.array([1,2,3,4,5])
block = np.cumsum(np.cumsum(Qi[:-1,:-1], 0), 1)

block[ka-1,ka-1]/(10*ka) + ka/(1-10)

In [ ]:
lcmc_k = lambda k : k/(1-10) + np.sum(Qi[:k,:k])/(10*k)
np.array([lcmc_k(k) for k in ka])

In [ ]:
for k in [1,2,3,4,5]:
    print(block[k-1,k-1],np.sum(Qi[:k,:k]))

In [ ]:
def compute_trustworthiness(Q_full: np.ndarray) -> np.ndarray:
    """
    T[k] for k = 1..m-1
    """
    Q = Q_full[:-1,:-1]

    m = Q.shape[0]
    T = np.zeros(m-1)
    for k in range(m-1):
        # 1 - normalized hard-k-intrusions. lower-left region. weighted by rank error (rank - k)
        Qs = Q[k:, :k]
        # a column vector of weights. weight = rank error = actual_rank - k
        W = np.arange(Qs.shape[0]).reshape(-1,1)
        denom = (k+1) * m * (m - k - 1)
        T[k] = 1 - np.sum(Qs * W) / denom
    return T


In [ ]:
def _trustworthiness(Q : np.ndarray, k_max = 'default') -> np.ndarray:
    # sec 3. https://www.sciencedirect.com/science/article/pii/S0925231209000101        
    
    N = Q.shape[0]
    Q_sliced = Q[:-1,:-1]
    
    m = N-1
    assert m == Q_sliced.shape[0]
    
    if k_max == 'default':
        k_lst = range(1,(N + 1) // 2)
    elif k_max == 'all':
        k_lst = range(1,N-1)
    else:
        raise ValueError('invalid k_max for trustworthiness curve')
    
    T = np.zeros(len(k_lst))
    true_rank = np.arange(1,N) #for W's
    
    for ix,k in enumerate(k_lst):
        # 1 - normalized hard-k-intrusions. lower-left region. weighted by rank error (rank - k)
        Qs = Q_sliced[k:, :k]
        
        # a column vector of weights. weight = rank error = actual_rank - k            
        W = (true_rank[k:]-k).reshape(-1,1)
            
        if k < N/2:
            denom = N*k*(2*N - 3*k - 1)            
        else:
            denom = N * (N-k) * (N-k-1)
        
        T[ix] = 1 - 2*np.sum(Qs * W) / denom
        
    return T


In [ ]:
# dist_X1 = pairwise_distances(sample_X)
# rankdata(dist_X1, axis=1, method="ordinal").astype(int)-1
# [np.argsort(np.argsort(row)) for row in dist_X1]

In [ ]:
# Q

In [ ]:
def coranking_matrix(R1, R2):
    '''
    Get co-ranking matrix

    Parameters
    ----------
    R1, R2 - two ranking matrices as input

    Return
    ------
    Q - coranking matrix
    '''


    R1 = np.array(R1)
    R2 = np.array(R2)
    assert R1.shape == R2.shape
    Q = np.zeros(R1.shape)
    m = len(Q)

    for i in range(m):
        for j in range(m):
            k = int(R1[i,j])
            l = int(R2[i,j])
            Q[k,l] += 1

    return Q


In [ ]:
nc_val = 2

X_pca = PCA(n_components=nc_val).fit_transform(sample_X)
# X_tsne = TSNE(n_components=nc_val, perplexity=30, random_state=0,method='barnes_hut').fit_transform(sample_X)

In [ ]:
from sklearn.manifold import trustworthiness

comps = [2, 3, 5, 10, 20]
ks = [5, 10, 20, 50, 100]
T = np.array([[trustworthiness(X, reducer(n_components=c).fit_transform(X), n_neighbors=k)
               for k in ks] for c in comps])

In [ ]:
X_pca_fit = PCA(n_components=nc_val).fit(sample_X)

In [ ]:
# inv_fit = X_pca_fit.inverse_transform(X_pca_fit)

In [ ]:
embedding = Isomap(n_components=2)
X_transformed = embedding.fit_transform(sample_X)

In [ ]:
# X_transformed

In [ ]:
trust_pca = trustworthiness(sample_X, X_pca, n_neighbors=5)
trust_tsne = trustworthiness(sample_X, X_tsne, n_neighbors=5)

print(f"PCA trustworthiness:  {trust_pca:.4f}")
print(f"t-SNE trustworthiness: {trust_tsne:.4f}")

In [ ]:
k = 3  # or any number based on your domain or elbow method
labels_pca = KMeans(n_clusters=k, random_state=0).fit_predict(X_pca)
labels_tsne = KMeans(n_clusters=k, random_state=0).fit_predict(X_tsne)


In [ ]:
ari = adjusted_rand_score(labels_pca, labels_tsne)
nmi = normalized_mutual_info_score(labels_pca, labels_tsne)

In [ ]:
nmi

In [ ]:
ari

In [ ]:

num_comps = [2,3,4,5]
pca_vec = []
tsne_vec = []

for nc_val in num_comps:

    X_pca = PCA(n_components=nc_val).fit_transform(sample_X)
    X_tsne = TSNE(n_components=nc_val, perplexity=30, random_state=0,method='exact').fit_transform(sample_X)

    k = 5  # or any number based on your domain or elbow method
    labels_pca = KMeans(n_clusters=k, random_state=0).fit_predict(X_pca)
    labels_tsne = KMeans(n_clusters=k, random_state=0).fit_predict(X_tsne)

    score_pca = silhouette_score(X_pca, labels_pca)
    score_tsne = silhouette_score(X_tsne, labels_tsne)

    pca_vec.append(score_pca)
    tsne_vec.append(score_tsne)

In [ ]:
from matplotlib import pyplot as plt
plt.scatter(num_comps,pca_vec)
plt.scatter(num_comps,tsne_vec)